In [1]:
import pandas as pd
import functions_simalign

/home/christopher/projects/corpus_tryout/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3291.82it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignore

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
df_1 = pd.read_csv("extraction_output/annotated_all_part_to_ger.csv").set_index("Unnamed: 0")
df_match_1 = pd.read_csv("extraction_output/adv_ger_matches.csv").set_index("Unnamed: 0")

In [4]:
df_1 = pd.concat([df_1, df_match_1[["best_match", "best_score"]]], axis=1)

In [5]:
df_1.loc[df_1["is_adv_participle"] & df_1["has_gerunziu"]].to_csv(
    "analysis/try_better_matching.csv"
)

In [6]:
df_1 = df_1.loc[df_1["is_adv_participle"] & df_1["has_gerunziu"]]

In [15]:
df_1.head(1)

,1,2,3,4,5,6,7,8,english,lemma,gerunzius_and_lemmas,is_adv_participle,has_gerunziu,best_match,best_score
Unnamed: 0,,,,,,,,,,,,,,,
27,carroll-alenka_v_kraji,"before her was another long passage , and the ...",hurrying,down it .,carroll-alenka_v_kraji,NaN,"înaintea ei se întindea un coridor lung, la ca...",NaN,"before her was another long passage, and the W...",hurry,"[('alergând', 'alerga')]",True,True,alergând,0.152291


In [ ]:
df_1["7"] = df_1["7"].str.replace("ş", "ș").replace("ţ", "ț")
df_1["gerunzius_and_lemmas"] = (
    df_1["gerunzius_and_lemmas"].str.replace("ş", "ș").replace("ţ", "ț") # TODO: this should happen at the very beginning
)

In [13]:
for _, row in df_1.head(1).iterrows():
    print(row["3"])

hurrying


In [50]:
df_1

In [64]:
matching_dict = {}
all_errors = []

In [ ]:
for i, row in df_1.iterrows():
    word_en = row["3"]
    sentence_en = row["english"]
    sentence_ro = row["7"]
    words_ro = [x[0] for x in eval(row["gerunzius_and_lemmas"])]
    truth_value, errors = functions_simalign.is_match(
        word_en=word_en,
        sentence_en=sentence_en,
        word_ro=words_ro,
        sentence_ro=sentence_ro,
    )
    if truth_value or (not truth_value and not errors):
        matching_dict[i] = truth_value
    if errors:
        for error in errors:
            print(f"{error.word=}, {error.sentence=}")
            all_errors.append(errors)

Could not find word -du-se!
error.word='-du-se', error.sentence='Cei trei oșteni umblară o clipă de colo-colo, uitân-du-se după vinovaţii ce trebuiau executaţi, apoi o luară la picior și se alăturară foarte liniștiţi convoiului.'
Could not find word -du-se!
error.word='-du-se', error.sentence='-Asta e foarte IMPORTANT, declară Regele, întorcân-du-se către juraţi.'
Could not find word -du-i-se!
error.word='-du-i-se', error.sentence='Şi totuși, mai știi, urmă, punând hârtia cu versuri pe genunchi și uitându-se chiorâș la ea, dacă stau să mă gândesc, parc-ar avea oarecare înţeles. " DOAR CĂ NU-ÎNOT NU-I BINE. " Nu știi să înoţi, nu-i așa? adăugă, adresân-du-i-se Valetului.'
Could not find word ţindu-mă!
error.word='ţindu-mă', error.sentence='Am coborât scara sim-ţindu-mă caraghioasă.'
Could not find word -nindu-se!
error.word='-nindu-se', error.sentence='-Antricot de mânzat, a răspuns ea spriji-nindu-se în mătură.'
Could not find word ţîndu-și!
error.word='ţîndu-și', error.sentence='Şi cu

KeyboardInterrupt: 

In [21]:
len(errors)

376

In [62]:
errors[25]

('shaking',
 'Frank gazed at each one at length, without comment, simply shaking his head.',
 ['-du'],
 'Frank se uitase atent și îndelung la fiecare în parte, fără comentarii, mulţumin-du-se să clatine din cap.')

In [43]:
functions_simalign.add_pronouns("întinzându", sentence='-Trebuie să-l pui și pe celălalt, a declarat el luând al doilea cercel și întinzându-mi-l.')

'întinzându-mi-l'

In [22]:
len(matching_dict)

4063

In [23]:
matching_df = pd.DataFrame(
    zip(matching_dict.keys(), matching_dict.values()),
    columns=["index", "part_matches_gerunziu"],
).set_index("index")

In [24]:
matching_df

,part_matches_gerunziu
index,
27,True
31,True
36,True
65,True
68,True
...,...
29389,True
29392,True
29396,True


In [26]:
df_with_matching = pd.concat([df_1, matching_df], axis=1)

In [29]:
df_with_matching["part_matches_gerunziu"].unique()

array([True, False, nan], dtype=object)

In [34]:
df_with_matching.loc[
    ~(df_with_matching["part_matches_gerunziu"].isna())
    & (df_with_matching["part_matches_gerunziu"] == False)
].sample(10).to_csv("analysis/adv_part_wrong_gerunziu_simalign_sample.csv")

In [ ]:
sentence = "Vorbele ei îi înspăimântă grozav pe cei prezenţi: câteva păsări o tăiară la fugă, în graba mare; o Coţofană bătrână se înfoie în pene, zicând:-Trebuie să pornesc spre casă, nu-mi priește aerul nopţii la gât ! Un Canar își chemă puișorii, zicându-le:-Veniţi iute, dragii mei !"
word = "așteptându"

In [33]:
functions_simalign.add_pronouns(word, sentence)

'zicându-le'

In [14]:
import re

In [15]:
re.split(';|,| ', "this, is a string to;, split")

['this', '', 'is', 'a', 'string', 'to', '', '', 'split']

In [18]:
word = "hurrying"
sentence = "before her was another long passage, and the White Rabbit was still in sight, hurrying down it."

In [21]:
splitting_chars = [" ", ",", ".", ":", ";"]
splitting_string = "|".join(splitting_chars)
words = [w for w in re.split(splitting_string, sentence) if w != ""]

In [25]:
splitting_string

' |,|.|:|;'

In [29]:
re.split(';|,| |\\.', sentence)

['before',
 'her',
 'was',
 'another',
 'long',
 'passage',
 '',
 'and',
 'the',
 'White',
 'Rabbit',
 'was',
 'still',
 'in',
 'sight',
 '',
 'hurrying',
 'down',
 'it',
 '']

In [24]:
re.split(splitting_string, sentence)

['',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '']

In [22]:
words

[]

In [ ]:
re.match(f".*({word}-?[a-zăâîșț]*).*", sentence).group(1)

In [44]:
test_sentence_en = "She return to the middle of the room, wondering how to get out of that place."
test_sentence_ro = "Se întoarse în mijlocul sălii, întrebându-și cum să facă să iasă din acel loc."

In [20]:
import re

In [21]:
word = "întrebându"

In [47]:
matching = re.match(f".*({word}-[a-zăâîșț]*) .*", test_sentence_ro)

In [48]:
matching.group(1)

'întrebându-și'

In [32]:
test_sentence_ro

'Se întoarse în mijlocul sălii, întrebându-se cum să facă să iasă din acel loc.'

In [35]:
print(re.findall("cum", test_sentence_ro))

['cum']


In [19]:
functions_simalign.aligner.get_word_aligns(test_sentence_en, test_sentence_ro)

{'mwmf': [(0, 0),
  (1, 1),
  (2, 2),
  (3, 3),
  (4, 3),
  (5, 3),
  (6, 4),
  (7, 4),
  (8, 5),
  (9, 6),
  (10, 7),
  (11, 10),
  (12, 10),
  (13, 11),
  (14, 12),
  (15, 13)],
 'inter': [(0, 0),
  (2, 2),
  (4, 3),
  (5, 3),
  (7, 4),
  (8, 5),
  (9, 6),
  (10, 7),
  (11, 10),
  (13, 11),
  (14, 12),
  (15, 13)],
 'itermax': [(0, 0),
  (1, 1),
  (2, 2),
  (3, 3),
  (4, 3),
  (5, 3),
  (6, 4),
  (7, 4),
  (8, 5),
  (9, 6),
  (10, 7),
  (10, 9),
  (11, 10),
  (12, 10),
  (13, 11),
  (14, 12),
  (15, 13)]}

In [4]:
df_1 = df_1[df_1["is_adv_participle"]] # only those are of our interest

In [5]:
len(df_1.loc[df_1["has_gerunziu"] & (df_1["best_score"] > 0.117)])

2348

In [6]:
df_1.loc[~(df_1["has_gerunziu"] & (df_1["best_score"] > 0.117))].sample(10).to_csv(
    "analysis/adv_part_no_gerunziu_sample.csv"
)

In [7]:
df_1.loc[(df_1["has_gerunziu"]) & ~(df_1["best_score"] > 0.117)].sample(20).to_csv(
    "analysis/adv_part_wrong_gerunziu_sample.csv"
)

In [8]:
df_1.loc[df_1["has_gerunziu"] & (df_1["best_score"] > 0.12)].to_csv("analysis/part_to_gerunziu.csv")

In [9]:
df_1.loc[~(df_1["has_gerunziu"] & df_1["best_score"] > 0.12)
].sample(20).to_csv("analysis/adv_part_no_gerunziu_sample.csv")

In [10]:
df_2 = pd.read_csv("extraction_output/annotated_all_ger_to_part.csv").set_index("Unnamed: 0")
df_match_2 = pd.read_csv("extraction_output/ger_adv_matches.csv").set_index("Unnamed: 0")

In [11]:
df_2 = pd.concat([df_2, df_match_2[["best_match", "best_score"]]], axis=1)

In [12]:
df_2 = df_2.loc[df_2["is_gerunziu"]] # only those are of our interest

In [13]:
len(df_2)

10413

In [14]:
df_2.loc[(df_2["has_adv_participle"])].head(10)

,1,2,3,4,5,6,7,8,romanian,lemma,adv_parts_and_lemmas,is_gerunziu,has_adv_participle,best_match,best_score
Unnamed: 0,,,,,,,,,,,,,,,
5,carroll-alenka_v_kraji,"înaintea ei se întindea un coridor lung , la c...",alergând,cât îl ţineau picioarele .,carroll-alenka_v_kraji,NaN,"before her was another long passage, and the W...",NaN,"înaintea ei se întindea un coridor lung, la ca...",alerga,"[('hurrying', 'hurry')]",True,True,hurrying,0.152291
7,carroll-alenka_v_kraji,"Şi , după ce- o străbătu în lung şi în lat ,",încercând,"în zadar fiecare uşă în parte , se întoarse în...",carroll-alenka_v_kraji,NaN,and when Alice had been all the way down one s...,NaN,"Şi, după ce-o străbătu în lung şi în lat, înce...",încerca,"[('wondering', 'wonder')]",True,True,wondering,0.035738
8,carroll-alenka_v_kraji,"Şi , după ce- o străbătu în lung şi în lat , î...",întrebându,-se cum să facă să iasă din acel loc .,carroll-alenka_v_kraji,NaN,and when Alice had been all the way down one s...,NaN,"Şi, după ce-o străbătu în lung şi în lat, înce...",întrebându,"[('wondering', 'wonder')]",True,True,wondering,0.153758
10,carroll-alenka_v_kraji,"şi ,",găsind,-o nemaipomenit de bună ( fiindcă avea un gust...,carroll-alenka_v_kraji,NaN,"However, this bottle was not marked ‘poison,’ ...",NaN,"şi, găsind-o nemaipomenit de bună (fiindcă ave...",găsi,"[('finding', 'find')]",True,True,finding,0.461690
14,carroll-alenka_v_kraji,"Nespus de frumos îmbrăcat ,",ţinând,într- o mână o pereche de mănuşi albe din piel...,carroll-alenka_v_kraji,NaN,"It was the White Rabbit returning, splendidly ...",NaN,"Nespus de frumos îmbrăcat, ţinând într-o mână ...",ţinca,"[('muttering', 'mutter')]",True,True,muttering,-0.042773
15,carroll-alenka_v_kraji,Venea tare grăbit şi,bombănind,întruna :,carroll-alenka_v_kraji,NaN,"It was the White Rabbit returning, splendidly ...",NaN,Venea tare grăbit şi bombănind întruna:,bombăni,"[('muttering', 'mutter')]",True,True,muttering,-2.000000
17,carroll-alenka_v_kraji,Alice adună de pe jos mănuşile şi evantaiul şi...,spunându,-şi :,carroll-alenka_v_kraji,NaN,"Alice took up the fan and gloves and, as the h...",NaN,Alice adună de pe jos mănuşile şi evantaiul şi...,spune,"[('talking', 'talk')]",True,True,talking,0.108255
22,carroll-alenka_v_kraji,""" Ce bine ar fi fost de n- aş fi plâns atât ! ...",înotând,şi sirăduindu -se să ajungă la mal .,carroll-alenka_v_kraji,NaN,"‘I wish I hadn’t cried so much !’ said Alice, ...",NaN,""" Ce bine ar fi fost de n-aş fi plâns atât ! ""...",înota,"[('trying', 'try')]",True,True,trying,0.033139
23,carroll-alenka_v_kraji,""" Ce bine ar fi fost de n- aş fi plâns atât ! ...",sirăduindu,-se să ajungă la mal .,carroll-alenka_v_kraji,NaN,"‘I wish I hadn’t cried so much !’ said Alice, ...",NaN,""" Ce bine ar fi fost de n-aş fi plâns atât ! ""...",sirădui,"[('trying', 'try')]",True,True,trying,-2.000000


In [15]:
len(df_2.loc[(df_2["has_adv_participle"]) & (df_2["best_score"] > 0.11)])

2873

In [18]:
df_2.loc[
    (df_2["has_adv_participle"])
    & (df_2["best_score"] > 0.11)
    & (df_2["best_score"] < 0.12)
].sample(10).to_csv("analysis/score_between_11_12_sample.csv")

In [1]:
df_2.loc[(df_2["has_adv_participle"]) & (df_2["best_score"] > 0.117)].to_csv(
    "analysis/gerunziu_to_part.csv"
)

NameError: name 'df_2' is not defined

In [ ]:
df_ger.loc[(df["has_adv_participle"]) & ~(df_ger["best_score"] > 0.12)].to_csv("analysis/gerund_translated_to")